# 第4周 · 周五：数据倾斜与查询优化

> 难度：⭐⭐⭐⭐⭐ | 定位：理解数据倾斜的本质，掌握检测、避免和解决倾斜问题的实战方法
> 前置依赖：周四的Hive分布式架构知识 | 后续用途：所有大数据量SQL的性能保障
> 面试地位：大厂数据分析面试高频题，约占数据工程类问题的20%

---

## 📘 区块1：核心概念与第一性原理

### 1.1 数据倾斜是什么——一个让你秒懂的比喻

你让10个人同时完成一个任务。任务被按key分配给每个人：
```
工人1：处理 user_1 ~ user_100     → 每人10条 → 共1000条
工人2：处理 user_101 ~ user_200   → 每人10条 → 共1000条
...
工人9：处理 user_801 ~ user_900   → 每人10条 → 共1000条
工人10：处理 user_901 + hot_user  → 100+1000000条
```

结果：9个工人1分钟做完，在玩手机。工人10花了1个小时。总耗时 = 1小时（等最慢的那个）。你加了10个工人也没用——因为那个hot_user的数据只能由一个人处理。

这就是**数据倾斜**：在分布式计算中，数据按key被分配到不同节点处理。如果某个key的数据量远超其他key，处理那个key的节点就会成为瓶颈，拖慢整个任务。

**核心认知**：加机器解决不了数据倾斜。10台机器里9台在发呆，1台在拼命——你加100台也只是101台发呆。

### 1.2 为什么数据倾斜是大数据环境下的"头号杀手"

在单机数据库（MySQL）中，数据倾斜不是什么大问题——反正所有数据都在一台机器上处理。

但在分布式环境（Hive/Spark）中，数据倾斜是性能问题的80%根因。因为分布式计算的核心机制是"按key分配任务"——这个机制假设key的分布是相对均匀的。当这个假设不成立时，整个并行架构的优势就荡然无存。

### 1.3 三种典型倾斜场景

**场景1：GROUP BY 倾斜**
```sql
-- 如果user_id='NULL'或某个'机器人'有1亿条记录
SELECT user_id, COUNT(*) FROM big_table GROUP BY user_id;
```
数据按user_id分配到Reduce节点，处理那个特殊user_id的Reduce任务会跑很久。

**场景2：JOIN 倾斜**
```sql
-- 如果orders表中user_id='hot_user'有1亿条，users表只有1条
SELECT * FROM orders o JOIN users u ON o.user_id = u.user_id;
```
Shuffle按user_id重分布数据时，hot_user的1亿条全被发往同一个Reduce。

**场景3：COUNT(DISTINCT) 倾斜**
```sql
SELECT COUNT(DISTINCT user_id) FROM big_table;
```
去重操作本质上需要把所有相同user_id的数据汇集到同一个节点——相当于强迫所有数据做一次全量Shuffle。

### 1.4 快速检测数据倾斜

**检测1：看任务进度**
- 大部分Map/Reduce任务完成了，但1-2个任务进度在99%卡住
- 这就是倾斜的典型表现

**检测2：看数据分布**
```sql
-- 快速检测key的分布是否均匀
SELECT key_field,
       COUNT(*) AS cnt,
       ROUND(COUNT(*) / SUM(COUNT(*)) OVER() * 100, 2) AS pct
FROM big_table
GROUP BY key_field
ORDER BY cnt DESC
LIMIT 10;
```
如果Top1占了超过30%的数据——有倾斜。如果Top1超过50%——严重倾斜。

**检测3：看执行计划中的Reducer数据量**
- 在Hive/Spark UI中，查看每个Reduce任务处理的数据量
- 如果最大值/平均值 > 3倍——有倾斜

### 1.5 面试高频考点

**面试官常问**：
- "什么是数据倾斜？如何发现？"
- "GROUP BY倾斜怎么处理？"
- "JOIN倾斜有哪些解决方法？"
- "为什么COUNT(DISTINCT)容易产生倾斜？怎么优化？"
- "加盐是什么意思？"

**回答要点**：
- 数据倾斜=某个key的数据量远超其他，导致单节点成为瓶颈
- 发现方式：看任务进度（99%卡住）、看key分布（Top1占比）、看Reducer数据量
- GROUP BY倾斜：过滤NULL、加盐（给key加随机后缀打散）、两阶段聚合
- JOIN倾斜：MAPJOIN（小表）、过滤倾斜key单独处理、加盐
- COUNT(DISTINCT)改写成先GROUP BY再COUNT，分两步走

## 🧩 区块2：知识点详细拆解

### 2.1 倾斜处理方法1：过滤NULL/无效值

**适用场景**：NULL值在分析中确实没有意义。

```sql
-- 如果NULL用户对你没有意义
SELECT user_id, SUM(amount) AS total_amount
FROM orders
WHERE user_id IS NOT NULL
GROUP BY user_id;
```

**但要小心**：不是所有NULL都没意义。比如订单金额的NULL可能代表"待定"而非常规值——盲目过滤会丢数据。

### 2.2 倾斜处理方法2：单独处理倾斜Key

**适用场景**：你知道是哪个具体的key导致了倾斜。

```sql
-- 把倾斜key单独拿出来处理，其余的按正常逻辑
-- 假设hot_user是倾斜的key
SELECT user_id, SUM(amount) AS total
FROM orders
WHERE user_id = 'hot_user'
GROUP BY user_id
UNION ALL
SELECT user_id, SUM(amount) AS total
FROM orders
WHERE user_id != 'hot_user'
GROUP BY user_id;
```

分开后，hot_user只需要处理自己的数据（虽然多但不受其他key干扰），其他key也不会被hot_user拖慢。

### 2.3 倾斜处理方法3：加盐——打散

**为什么重要**：加盐是最常用的倾斜处理方法，适合"不知道具体哪个key倾斜"或"多个key倾斜"的场景。

**加盐的原理**：给key加上随机后缀，把一个大key人为拆散成多个小key，先局部聚合，再全局聚合。

```sql
-- 原始（倾斜严重）
SELECT user_id, COUNT(*) AS cnt
FROM big_table
GROUP BY user_id;

-- 加盐（打散倾斜）
SELECT 
    REGEXP_REPLACE(salted_key, '_[0-9]+$', '') AS user_id,
    SUM(partial_cnt) AS cnt
FROM (
    SELECT 
        CONCAT(user_id, '_', CAST(RAND() * 10 AS INT)) AS salted_key,
        COUNT(*) AS partial_cnt
    FROM big_table
    GROUP BY CONCAT(user_id, '_', CAST(RAND() * 10 AS INT))
) t
GROUP BY REGEXP_REPLACE(salted_key, '_[0-9]+$', '');
```

**加盐的代价**：
- 两阶段聚合：多一层子查询
- 需要去掉盐值恢复原始key：用正则或字符串函数
- 代码更复杂了——但跑得动了

**加盐的适用条件**：
- GROUP BY场景下key分布严重不均
- JOIN场景下大表有倾斜key
- 盐值的范围要合理：太少了打不散，太多了计算量暴增（经验值：10-100）

### 2.4 替代COUNT(DISTINCT)

**为什么重要**：COUNT(DISTINCT)是性能杀手——它把所有数据汇集到一个节点做去重。

```sql
-- 慢写法（所有数据汇集到一个节点去重）
SELECT category, COUNT(DISTINCT user_id) AS uv
FROM orders
GROUP BY category;

-- 快写法（先按user_id+categor分组去重，再count）
SELECT category, COUNT(*) AS uv
FROM (
    SELECT user_id, category
    FROM orders
    GROUP BY user_id, category
) t
GROUP BY category;
```

**原理**：
- 慢写法：去重操作需要把所有相同user_id的行的category信息汇集到一起→严重Shuffle
- 快写法：先GROUP BY user_id+category（局部去重，可以并行），再COUNT（轻量聚合）

**性能差异**：在100亿行数据上，前者可能跑1小时，后者可能跑5分钟。

### 2.5 JOIN倾斜的处理方案

**方案1：MAPJOIN（最优方案）**
如果一张表足够小——直接MAPJOIN，消除Shuffle：
```sql
SELECT /*+ MAPJOIN(b) */ ...
FROM big_table a JOIN small_table b ON a.key = b.key;
```

**方案2：过滤倾斜Key（单独处理）**
```sql
SELECT ... FROM big_table a JOIN small_table b ON a.key = b.key
WHERE a.key != 'hot_key'
UNION ALL
SELECT ... FROM big_table a JOIN small_table b ON a.key = b.key
WHERE a.key = 'hot_key';
```

**方案3：加盐JOIN**
给大表的倾斜key加上随机后缀，小表的对应key复制N份：
```sql
-- 小表扩容（对倾斜key复制10份）
WITH expanded_small AS (
    SELECT key, value, 0 AS salt FROM small_table WHERE key != 'hot_key'
    UNION ALL
    SELECT key, value, salt 
    FROM small_table 
    CROSS JOIN (SELECT explode(array(0,1,2,3,4,5,6,7,8,9)) AS salt) s
    WHERE key = 'hot_key'
),
-- 大表加盐
salted_big AS (
    SELECT 
        CASE WHEN key = 'hot_key' 
             THEN CONCAT(key, '_', CAST(RAND()*10 AS INT))
             ELSE key 
        END AS salted_key,
        key AS original_key,
        amount
    FROM big_table
)
SELECT ...
FROM salted_big a 
JOIN expanded_small b ON a.salted_key = b.key;
```

### 2.6 数据倾斜的预防

**为什么重要**：预防比事后处理更高效。

**预防策略**：
1. **表设计阶段**：选择分布均匀的列作为分区键
2. **分桶（Bucketing）**：在建表时让指定列的数据均匀分布到固定数量的桶中
   ```sql
   CREATE TABLE orders_bucketed (
       order_id BIGINT, user_id BIGINT, amount DECIMAL)
   CLUSTERED BY (user_id) INTO 256 BUCKETS;
   ```
3. **数据质量监控**：定期检查key分布，发现倾斜及时处理
4. **在数据入仓时加盐**：在DWD层就对已知的倾斜key做打散处理

### 2.7 数据倾斜 vs 数据量大——分类处理

| 症状 | 原因 | 解决方案 |
|------|------|----------|
| 所有Task都慢 | 数据量大 | 加资源、优化SQL、分区裁剪 |
| 大部分Task快，1-2个慢 | 数据倾斜 | 过滤/拆分/加盐/改COUNT DISTINCT |
| 最后阶段特别慢 | 可能倾斜+Shuffle | 检查Reduce数量和key分布 |
| Map阶段慢 | 扫描量大 | 分区裁剪、列裁剪 |
| Reduce阶段慢 | 倾斜或Reduce数太少 | 增大Reduce数、检查倾斜 |

### 2.8 实际工作中的倾斜处理流程

```
第一步：确认是不是倾斜
  → 看任务界面（MapReduce UI / Spark UI）
  → 看99%卡住的Task处理的数据量是否远大于其他

第二步：定位倾斜的Key
  → SELECT key, COUNT(*) FROM table GROUP BY key ORDER BY COUNT(*) DESC LIMIT 20;
  → 看Top key的占比

第三步：选择处理策略
  → NULL值没意义 → 直接过滤
  → 明确的几个倾斜key → 单独处理（UNION ALL）
  → 大量key都倾斜或不明确是哪个 → 加盐
  → COUNT(DISTINCT) → 改写为先GROUP BY再COUNT
  → JOIN倾斜 → MAPJOIN或加盐JOIN

第四步：验证效果
  → 确认优化后查询能跑完且耗时可接受
  → 记录优化前后的数据量和耗时对比
```

## 💻 区块3：动手练习

### 步骤0：准备——模拟数据倾斜场景

In [ ]:
# ===== 步骤0：模拟数据倾斜 =====
import numpy as np
np.random.seed(42)

print("=" * 60)
print("模拟数据倾斜——创建分布严重不均的数据")
print("=" * 60)
print()

# 模拟：100万行数据，其中'hot_user'占50%，其余分散在10万个用户
n = 1_000_000
user_ids = np.random.choice(
    ["hot_user"] + [f"user_{i}" for i in range(1, 100000)],
    n,
    p=[0.5] + [0.5/99999]*99999
)

from collections import Counter
counter = Counter(user_ids)
top_10 = counter.most_common(10)

print("数据分布分析：")
print(f"  总行数：{n:,}")
print(f"  唯一用户数：{len(counter):,}")
print()
print("Top 10用户的行数占比：")
for uid, count in top_10:
    pct = count / n * 100
    bar = '█' * int(pct * 2)
    print(f"  {uid:<15} {count:>8,} 行 ({pct:5.1f}%) {bar}")
print()

# 计算倾斜程度
top1_pct = top_10[0][1] / n * 100
print(f"倾斜评估：Top1占比 = {top1_pct:.1f}%")
if top1_pct > 30:
    print("  结论：严重倾斜！任何GROUP BY user_id的操作都会卡死。")
    print("  如果不处理：9个Reduce各处理几十行，1个Reduce处理50万行。")
    print("  等了1小时，那9个Reduce早就完了，最后一个还在跑。")


### 步骤1：基础练习——倾斜检测SQL

In [ ]:
# ===== 步骤1：倾斜检测SQL模板 =====

print("=" * 60)
print("标准倾斜检测SQL——每次跑大查询前先跑这个")
print("=" * 60)
print()

print("-- 检测1：查看key分布")
print("SELECT ")
print("    key_field,")
print("    COUNT(*) AS cnt,")
print("    ROUND(COUNT(*) / SUM(COUNT(*)) OVER() * 100, 2) AS pct")
print("FROM big_table")
print("WHERE dt = '2026-01-15'")
print("GROUP BY key_field")
print("ORDER BY cnt DESC")
print("LIMIT 20;")
print()

print("-- 检测2：检查NULL值占比")
print("SELECT")
print("    SUM(CASE WHEN key_field IS NULL THEN 1 ELSE 0 END) AS null_cnt,")
print("    COUNT(*) AS total_cnt,")
print("    ROUND(SUM(CASE WHEN key_field IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS null_pct")
print("FROM big_table")
print("WHERE dt = '2026-01-15';")
print()

print("-- 检测3：检查JOIN key的倾斜（两个表都要看）")
print("SELECT")
print("    'orders表' AS table_name,")
print("    user_id,")
print("    COUNT(*) AS cnt")
print("FROM orders")
print("WHERE dt = '2026-01-15'")
print("GROUP BY user_id")
print("ORDER BY cnt DESC")
print("LIMIT 10;")


### 步骤2：进阶练习——优化方法对比

In [ ]:
# ===== 步骤2：三种倾斜处理方法的对比 =====

print("=" * 60)
print("场景：统计每个用户的订单数（user_id严重倾斜）")
print("=" * 60)
print()

print("【方法0：原始写法——会挂】")
print("-" * 40)
print("SELECT user_id, COUNT(*) AS cnt")
print("FROM orders")
print("WHERE dt = '2026-01-15'")
print("GROUP BY user_id;")
print("→ 如果hot_user占50%数据→严重倾斜→可能1小时跑不完")
print()

print("【方法1：过滤NULL——如果NULL是你不需要的】")
print("-" * 40)
print("SELECT user_id, COUNT(*) AS cnt")
print("FROM orders")
print("WHERE dt = '2026-01-15'")
print("  AND user_id IS NOT NULL")
print("GROUP BY user_id;")
print("→ 只对NULL有效，对hot_user无效")
print()

print("【方法2：单独处理倾斜key】")
print("-" * 40)
print("SELECT user_id, COUNT(*) AS cnt")
print("FROM orders")
print("WHERE dt = '2026-01-15' AND user_id = 'hot_user'")
print("GROUP BY user_id")
print("UNION ALL")
print("SELECT user_id, COUNT(*) AS cnt")
print("FROM orders")
print("WHERE dt = '2026-01-15' AND user_id != 'hot_user'")
print("GROUP BY user_id;")
print("→ 需要事先知道哪个key倾斜")
print()

print("【方法3：加盐——两阶段聚合】")
print("-" * 40)
print("SELECT")
print("    REGEXP_REPLACE(salted_key, '_[0-9]+$', '') AS user_id,")
print("    SUM(partial_cnt) AS cnt")
print("FROM (")
print("    SELECT")
print("        CONCAT(user_id, '_', CAST(RAND()*10 AS INT)) AS salted_key,")
print("        COUNT(*) AS partial_cnt")
print("    FROM orders")
print("    WHERE dt = '2026-01-15'")
print("    GROUP BY CONCAT(user_id, '_', CAST(RAND()*10 AS INT))")
print(") t")
print("GROUP BY REGEXP_REPLACE(salted_key, '_[0-9]+$', '');")
print("→ 通用方案，但代码更复杂")
print()

print("选择策略：")
print("  ✓ 知道具体倾斜key → 方法2（单独处理）")
print("  ✓ 不知道具体倾斜key → 方法3（加盐）")
print("  ✓ 倾斜的是NULL且不需要NULL → 方法1（简单过滤）")
print("  ✓ 倾斜的是大表JOIN → MAPJOIN或加盐JOIN")


### 步骤3：挑战练习——综合场景

In [ ]:
# ===== 步骤3：诊断一个真实的慢查询场景 =====

print("=" * 60)
print("场景：电商618大促——同样的SQL昨天跑了2分钟，今天2小时还没出")
print("=" * 60)
print()

print("SQL:")
print("  SELECT user_id, SUM(amount), COUNT(DISTINCT product_id)")
print("  FROM orders")
print("  WHERE dt = '2026-06-18'")
print("  GROUP BY user_id;")
print()

print("诊断步骤：")
print()
print("第1步：检查数据量")
print("  昨天(dt=2026-06-17): 2000万行")
print("  今天(dt=2026-06-18): 2亿行 → 10倍增长")
print()
print("第2步：检查key分布")
print("  Top1 user_id = 'VIP_888' → 占15%的数据（3000万行）")
print("  昨天这个用户只占0.5% → 数据倾斜大大加剧")
print()
print("第3步：确认问题")
print("  问题1：COUNT(DISTINCT product_id) → 需要额外Shuffle")
print("  问题2：VIP_888倾斜严重 → 单节点处理3000万行")
print("  问题3：618当天有大量'刷单机器人'——user_id=NULL占10%")
print()
print("第4步：优化方案")
print("  1. 过滤NULL user_id（机器人）")
print("  2. 改写COUNT(DISTINCT)为GROUP BY子查询")
print("  3. 对VIP_888单独处理（UNION ALL）")
print()
print("优化后预估：2小时 → 5分钟")


## 📊 区块4：业务场景应用

### 场景1：为什么昨天同样的SQL跑得飞快，今天挂掉了

昨天是普通工作日，今天是618。

SQL：
```sql
SELECT user_id, SUM(amount) AS total_amount
FROM orders 
WHERE dt = '2026-06-18' 
GROUP BY user_id;
```

平时每天订单量2000万，正常运行。618订单量2亿——数据量涨了10倍。更糟糕的是，出现了大量"超级买家"——一个user_id下几百单。倾斜也变严重了。

同样的SQL、同样的集群、同样的优化——但数据变了。

**排查**：先看数据量（2亿 vs 平时2000万），再看key分布（VIP_888占15%），确认是"数据量增大+倾斜加剧"的叠加效应。

**处理**：单独处理VIP_888 + 对剩余数据加盐 + 增加Reduce并行度。

**关键启示**：SQL性能不是静态的——它取决于数据。大促、节假日、活动期间，平时的SQL可能跑不动。你要有"数据量敏感性"。

### 场景2：COUNT(DISTINCT)拖垮了整个集群

运营要"看今天有多少个不同用户下了单"——你写了：
```sql
SELECT COUNT(DISTINCT user_id) FROM orders WHERE dt = '2026-01-15';
```
这个看似简单的SQL跑了30分钟还没出，还影响了其他同时在跑的任务。

为什么？`COUNT(DISTINCT user_id)`在Hive中展开为一个全量Shuffle+去重操作——所有数据都被汇聚到少数几个Reducer做去重，计算量和网络传输都很密集。

**优化**：
```sql
SELECT COUNT(*) FROM (
    SELECT user_id FROM orders 
    WHERE dt = '2026-01-15' 
    GROUP BY user_id
) t;
```
先GROUP BY在每个节点做局部去重（可以并行），再COUNT（轻量）。

**效果**：30分钟 → 3分钟。

### 场景3：用户画像表的倾斜噩梦

你要生成一张"用户画像宽表"：
```sql
SELECT 
    u.user_id,
    u.city,
    u.level,
    SUM(o.amount) AS total_amount,
    COUNT(DISTINCT o.product_id) AS product_diversity
FROM dim_user u
LEFT JOIN orders o ON u.user_id = o.user_id
WHERE o.dt BETWEEN '2026-01-01' AND '2026-06-30'
GROUP BY u.user_id, u.city, u.level;
```
这个SQL跑了2小时。问题在哪？
1. orders表可能有某些"机器人用户"占了大量订单→JOIN倾斜
2. COUNT(DISTINCT product_id)→去重Shuffle
3. 半年的订单量巨大（可能数十亿行）

**优化方案**：先预聚合orders表（按user_id GROUP BY），消除倾斜和去重瓶颈，再JOIN用户维表。

### 场景4：NULL值带来的隐秘倾斜

你的SQL跑得很慢但你以为'数据量本来就这么大'：
```sql
SELECT channel_id, COUNT(*) FROM orders GROUP BY channel_id;
```
跑了一个30分钟的检查：
```sql
SELECT channel_id, COUNT(*) AS cnt FROM orders GROUP BY channel_id ORDER BY cnt DESC LIMIT 10;
```
结果发现：`channel_id IS NULL`占了40%的数据！这些是App早期版本的订单，没有记录渠道ID。

处理：如果渠道分析不需要这40%的NULL数据→直接过滤。如果需要→单独处理（把NULL标记为'unknown'）。

**关键启示**：数据倾斜不总是"一个大key"——有时候"一堆NULL"也是倾斜的根源。

## ✅ 区块5：分级检验清单

### 🟢 初级检验（必须能独立完成的）
- [ ] 能用白话解释什么是数据倾斜
- [ ] 能识别SQL中的三个倾斜风险点（GROUP BY / JOIN / COUNT DISTINCT）
- [ ] 能写出检查key分布的SQL
- [ ] 知道COUNT(DISTINCT)的优化写法

**答案提示**：
- 数据倾斜=某个key的数据量特别大，导致分布式计算中单个节点成为瓶颈
- GROUP BY倾斜（key分布不均）、JOIN倾斜（key匹配不均）、COUNT(DISTINCT)倾斜（全量去重Shuffle）
- 检查SQL：SELECT key, COUNT(*) ... GROUP BY key ORDER BY COUNT(*) DESC LIMIT 20
- COUNT(DISTINCT x) → SELECT COUNT(*) FROM (SELECT x FROM t GROUP BY x) tmp

### 🟡 中级检验（能解释给同事听的原理）
- [ ] 能解释加盐的原理和适用场景
- [ ] 能把COUNT(DISTINCT)改写为GROUP BY版本
- [ ] 能用三种方法处理GROUP BY倾斜（过滤/拆分/加盐）
- [ ] 遇到查询慢时会先怀疑倾斜而不是加资源

**答案提示**：
- 加盐=给key加随机后缀打散→先局部聚合→去掉盐值→再全局聚合
- 过滤(NULL不需要时)、拆分(知道具体倾斜key时)、加盐(不知道具体倾斜key时)
- 慢查询排查顺序：数据量→key分布→分区裁剪→JOIN方式→并行度

### 🔴 高级检验（能举一反三的迁移能力）
- [ ] 能根据数据分布动态选择倾斜处理策略
- [ ] 能在JOIN场景下设计加盐方案（大表加盐+小表复制）
- [ ] 能设计表的分桶策略来预防倾斜
- [ ] 能在大促/活动前预判倾斜风险并提前处理

**答案提示**：
- 策略选择：NULL→过滤，已知倾斜key→拆分，未知倾斜→加盐，JOIN→MAPJOIN/加盐JOIN
- 分桶=在建表时就将数据均匀分布到N个桶中，按桶做采样和JOIN可以缓解倾斜
- 大促预见：预估数据量增长→预估key分布变化→提前重写SQL→提前测试

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 昨日** | Hive分布式架构 | 理解了MapReduce/Spark的并行机制，才能理解为什么倾斜会"拖死"整个任务 |
| **← 周三** | SCD Type 2 | Type 2导致的表膨胀（一个用户多行）可能加剧JOIN时的倾斜 |
| **← 周二** | 数仓分层 | DWD/DWS/ADS不同层的表特点不同——DWD易倾斜（明细）、DWS不易（预聚合） |
| **→ 明日** | 数仓综合练习 | 综合运用分层、SCD、Hive优化和倾斜处理设计完整方案 |
| **→ 第5-6周** | SQL分析实战 | 写的每个大SQL都要评估倾斜风险 |
| **业务连接** | 大促保障 | 618/双十一前对核心SQL做倾斜检测和优化 |

### 横向连接

**与平台运维的关联**：
- 如果经常倾斜→需要推动运维建立倾斜监控告警
- 大促期间需要运维预留更多资源应对倾斜

**与数据治理的关联**：
- 定期检查关键表的key分布应纳入数据质量监控
- 倾斜也可以作为数据质量的指标之一

**与成本控制的关联**：
- 严重倾斜浪费大量计算资源（9个节点在发呆）
- 处理倾斜=直接省钱

## 📝 区块7：当日小结

> **数据倾斜是分布式计算的"头号杀手"——10个节点9个在发呆、1个在拼命，加更多机器也解决不了问题。发现倾斜靠三招：看任务进度（有没有99%卡住的）、看key分布（Top1占比多少）、看Reducer数据量（最大值/平均值）。治理倾斜靠四招：过滤（NULL不需要时）、拆分（单独处理倾斜key）、加盐（打散倾斜key）、改写（COUNT(DISTINCT) → GROUP BY + COUNT）。记住：一个SQL跑不动，80%的可能是因为数据倾斜。先检查是不是某个key太"大"了，再考虑加资源。**

**三条金句带回家**：
1. 加机器解决不了数据倾斜——9个人在发呆，加100个也是101个人在发呆
2. COUNT(DISTINCT)是性能杀手——改写成先GROUP BY再COUNT，从1小时变5分钟
3. 大促之前不要等SQL挂了再行动——提前做倾斜检测和SQL优化

### 🔖 课后拓展

1. **练习**：找到你日常中跑得最慢的3条SQL，用倾斜检测方法检查key分布，判断是否存在倾斜

2. **思考**：如果数据倾斜无法完全消除（比如双十一的"热门商品"就是事实存在的），在不改SQL的前提下，有没有可能从平台层面缓解？

3. **预习**：明天的综合练习会用到本周所有知识——设计一个包含分层+SCD+优化+防倾斜的数仓方案

4. **面试准备**：背诵以下面试题答案：
   - "什么是数据倾斜？怎么发现？"
   - "GROUP BY倾斜怎么处理？"
   - "COUNT(DISTINCT)为什么慢？怎么优化？"
   - "加盐是什么？什么场景用它？"

---

*03analyst_thinking · Week 4 Day 5 · 数据倾斜与查询优化*